# 01 — Bronze to Silver ETL

**Course:** Big Data Management Systems and Tools  
**Project:** Batch vs. Streaming Analytics at Scale (Apache Spark)  
**Dataset:** City of Chicago — Traffic Crashes – Crashes (Socrata ID: 85ca-t3if)

---

## Purpose

This notebook implements the **Bronze → Silver** transformation. It is part of a [Lambda Architecture](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html) where:

- **Bronze** = raw, unchanged CSV as downloaded from the City of Chicago Open Data Portal.
- **Silver** = cleaned, canonicalized Parquet — the single source of truth for all downstream analysis.
- **Gold** = batch and streaming aggregation outputs (produced by later notebooks/scripts).

**Run this notebook once per dataset snapshot.** All downstream notebooks (`02_batch_analysis`, `03_streaming_logic_preview`) and scripts (`make_replay_files.py`, `streaming_job.py`) read from Silver and must NOT re-run this step.

---

## Scope

- Years: **2022–2025 inclusive**
- Rows outside this range are dropped during this step.

---

## What this notebook does (no analytics, no aggregations)

1. Start a local Spark session.
2. Read the raw CSV (Bronze) without schema inference.
3. Select the 9 canonical columns; drop all others.
4. Parse `CRASH_DATE` to a proper timestamp.
5. Cast injury columns to integers.
6. Filter to 2022–2025.
7. Derive `year` and `month` partition columns.
8. Write partitioned Parquet to `data/silver/crashes_parquet/`.
9. Validate the output with a read-back check.

In [1]:
import os
import findspark
from pyspark.sql import SparkSession

# ── Spark / Java environment ──────────────────────────────────────────────────
# Java 11 must be installed. Adjust the path below if your JDK lives elsewhere.
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-11.0.30.7-hotspot"

# ── Hadoop / winutils (Windows only) ─────────────────────────────────────────
# On Windows, Spark needs winutils.exe + hadoop.dll to write to the local filesystem.
# Binaries live at C:\hadoop\bin\ (hadoop-3.3.5, compatible with Spark 3.4.0).
# HADOOP_HOME tells Spark where to find winutils.exe.
# C:\hadoop\bin must also be on PATH so the JVM can load hadoop.dll via JNI.
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = r"C:\hadoop\bin" + ";" + os.environ.get("PATH", "")

findspark.init()

# ── Path constants ─────────────────────────────────────────────────────────────
# All paths are relative to the notebook directory so the project is portable.
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))
PROJECT_ROOT = os.path.join(NOTEBOOK_DIR, "..")

BRONZE_PATH = os.path.join(PROJECT_ROOT, "data", "raw", "Traffic_Crashes_-_Crashes.csv")
SILVER_PATH = os.path.join(PROJECT_ROOT, "data", "silver", "crashes_parquet")

print(f"Bronze source : {os.path.abspath(BRONZE_PATH)}")
print(f"Silver target : {os.path.abspath(SILVER_PATH)}")

# ── SparkSession ───────────────────────────────────────────────────────────────
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("bronze_to_silver")
    .config("spark.driver.memory", "12g")
    .config("spark.sql.execution.arrow.pyspark.enabled", True)
    .config("spark.sql.repl.eagerEval.enabled", True)
    .getOrCreate()
)

print(f"Spark version : {spark.version}")

Bronze source : c:\Users\miguel.morales\GIT\AI Certificate\Big Data\big-data_group-10\data\raw\Traffic_Crashes_-_Crashes.csv
Silver target : c:\Users\miguel.morales\GIT\AI Certificate\Big Data\big-data_group-10\data\silver\crashes_parquet
Spark version : 3.4.0


## Step 1 — Read Bronze (Raw CSV)

The **Bronze layer** is the raw data exactly as downloaded — no modifications, no type coercion.

- `inferSchema=False` keeps every column as a plain string. This avoids Spark incorrectly parsing ambiguous values (e.g., mixed date formats, missing numerics) at read time. We will cast types explicitly in the cleaning step.
- A `header=True` read picks up the column names from the first row.

We verify the read with a row count and a quick preview before touching any data.

In [2]:
# Read raw CSV — all columns as strings, no schema inference
bronze_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .option("multiLine", "true")   # handles any quoted fields with newlines
    .option("escape", '"')
    .csv(BRONZE_PATH)
)

bronze_row_count = bronze_df.count()
print(f"Bronze row count  : {bronze_row_count:,}")
print(f"Bronze column count: {len(bronze_df.columns)}")
bronze_df.printSchema()

Bronze row count  : 1,044,479
Bronze column count: 48
root
 |-- CRASH_RECORD_ID: string (nullable = true)
 |-- CRASH_DATE_EST_I: string (nullable = true)
 |-- CRASH_DATE: string (nullable = true)
 |-- POSTED_SPEED_LIMIT: string (nullable = true)
 |-- TRAFFIC_CONTROL_DEVICE: string (nullable = true)
 |-- DEVICE_CONDITION: string (nullable = true)
 |-- WEATHER_CONDITION: string (nullable = true)
 |-- LIGHTING_CONDITION: string (nullable = true)
 |-- FIRST_CRASH_TYPE: string (nullable = true)
 |-- TRAFFICWAY_TYPE: string (nullable = true)
 |-- LANE_CNT: string (nullable = true)
 |-- ALIGNMENT: string (nullable = true)
 |-- ROADWAY_SURFACE_COND: string (nullable = true)
 |-- ROAD_DEFECT: string (nullable = true)
 |-- REPORT_TYPE: string (nullable = true)
 |-- CRASH_TYPE: string (nullable = true)
 |-- INTERSECTION_RELATED_I: string (nullable = true)
 |-- NOT_RIGHT_OF_WAY_I: string (nullable = true)
 |-- HIT_AND_RUN_I: string (nullable = true)
 |-- DAMAGE: string (nullable = true)
 |-- DATE_

## Step 2 — Select Canonical Columns (Silver Schema)

The raw dataset contains many columns irrelevant to our analysis (vehicle details, geographic coordinates, administrative codes, etc.). We drop them immediately to reduce shuffle cost during writes and to keep all downstream code simple.

**Retained columns and their roles:**

| Column | Role |
|---|---|
| `CRASH_RECORD_ID` | Traceability — never used for grouping or aggregation |
| `CRASH_DATE` | Primary event timestamp — all time-based analysis uses this |
| `INJURIES_TOTAL` | Main severity metric |
| `INJURIES_FATAL` | Fatal subset — used in severity aggregations |
| `MOST_SEVERE_INJURY` | Categorical severity label |
| `WEATHER_CONDITION` | Environmental context |
| `LIGHTING_CONDITION` | Environmental context |
| `ROADWAY_SURFACE_COND` | Environmental context |
| `PRIM_CONTRIBUTORY_CAUSE` | Primary cause label — used in cause-over-time analysis |
| `SEC_CONTRIBUTORY_CAUSE` | Secondary cause label |

All other columns are dropped here and will not appear in Silver or Gold.

In [3]:
# Canonical column list — locked by project specification
CANONICAL_COLUMNS = [
    "CRASH_RECORD_ID",
    "CRASH_DATE",
    "INJURIES_TOTAL",
    "INJURIES_FATAL",
    "MOST_SEVERE_INJURY",
    "WEATHER_CONDITION",
    "LIGHTING_CONDITION",
    "ROADWAY_SURFACE_COND",
    "PRIM_CONTRIBUTORY_CAUSE",
    "SEC_CONTRIBUTORY_CAUSE",
]

# Validate that every expected column exists in the raw file before selecting
missing = [c for c in CANONICAL_COLUMNS if c not in bronze_df.columns]
if missing:
    raise ValueError(f"Missing columns in Bronze CSV: {missing}")

# Drop all columns not in the canonical list
slim_df = bronze_df.select(CANONICAL_COLUMNS)

print(f"Columns retained : {len(slim_df.columns)}")
print(f"Columns dropped  : {len(bronze_df.columns) - len(slim_df.columns)}")
slim_df.show(3, truncate=50)

Columns retained : 10
Columns dropped  : 38
+--------------------------------------------------+----------------------+--------------+--------------+-----------------------+-----------------+------------------+--------------------+---------------------------+---------------------------+
|                                   CRASH_RECORD_ID|            CRASH_DATE|INJURIES_TOTAL|INJURIES_FATAL|     MOST_SEVERE_INJURY|WEATHER_CONDITION|LIGHTING_CONDITION|ROADWAY_SURFACE_COND|    PRIM_CONTRIBUTORY_CAUSE|     SEC_CONTRIBUTORY_CAUSE|
+--------------------------------------------------+----------------------+--------------+--------------+-----------------------+-----------------+------------------+--------------------+---------------------------+---------------------------+
|000c4307d8e9b39075cffdd0aade3603e0f96f14e41da9e...|01/14/2025 12:25:00 PM|             0|             0|NO INDICATION OF INJURY|             SNOW|          DAYLIGHT|       SNOW OR SLUSH| IMPROPER TURNING/NO SIGNAL|IMPROPER 

## Step 3 — Parse, Clean, and Filter

Minimal cleaning rules — only what is necessary to make the data usable. No imputation, no business-logic transformations.

**Rules applied:**

1. **Parse `CRASH_DATE`** — The Chicago dataset format is `MM/dd/yyyy hh:mm:ss a` (12-hour clock with AM/PM). We use `to_timestamp` with the explicit pattern. Rows where parsing fails become `null` and are dropped.
2. **Cast injury columns** — `INJURIES_TOTAL` and `INJURIES_FATAL` are cast from string to integer. Non-numeric values (e.g., empty strings) become `null`.
3. **Drop rows with null `CRASH_DATE`** — The timestamp is the foundation of every downstream analysis; rows without it are unusable.
4. **Filter 2022–2025** — Rows outside the project scope are dropped.
5. **Derive `year` and `month`** — Integer columns extracted from `CRASH_DATE` and used as Parquet partition keys. They are NOT redundant — Spark uses them for partition pruning in downstream reads.

In [4]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

# ── 1. Parse CRASH_DATE ───────────────────────────────────────────────────────
# Expected format in the Chicago dataset: "01/05/2022 08:30:00 PM"
CRASH_DATE_FORMAT = "MM/dd/yyyy hh:mm:ss a"

cleaned_df = slim_df.withColumn(
    "CRASH_DATE",
    F.to_timestamp(F.col("CRASH_DATE"), CRASH_DATE_FORMAT)
)

# ── 2. Cast injury columns to integer ─────────────────────────────────────────
cleaned_df = cleaned_df \
    .withColumn("INJURIES_TOTAL", F.col("INJURIES_TOTAL").cast(IntegerType())) \
    .withColumn("INJURIES_FATAL", F.col("INJURIES_FATAL").cast(IntegerType()))

# ── 3. Drop rows where CRASH_DATE could not be parsed ────────────────────────
pre_null_drop = cleaned_df.count()
cleaned_df = cleaned_df.filter(F.col("CRASH_DATE").isNotNull())
null_dropped = pre_null_drop - cleaned_df.count()
print(f"Rows dropped (null CRASH_DATE) : {null_dropped:,}")

# ── 4. Derive year/month and filter to 2022–2025 ─────────────────────────────
cleaned_df = cleaned_df \
    .withColumn("year",  F.year(F.col("CRASH_DATE"))) \
    .withColumn("month", F.month(F.col("CRASH_DATE")))

pre_filter_count = cleaned_df.count()
silver_df = cleaned_df.filter(
    (F.col("year") >= 2022) & (F.col("year") <= 2025)
)
post_filter_count = silver_df.count()

print(f"Rows before year filter : {pre_filter_count:,}")
print(f"Rows after  year filter : {post_filter_count:,}")
print(f"Rows dropped (out of scope) : {pre_filter_count - post_filter_count:,}")

# ── Preview ───────────────────────────────────────────────────────────────────
silver_df.printSchema()
silver_df.show(5, truncate=40)

Rows dropped (null CRASH_DATE) : 0
Rows before year filter : 1,044,479
Rows after  year filter : 440,330
Rows dropped (out of scope) : 604,149
root
 |-- CRASH_RECORD_ID: string (nullable = true)
 |-- CRASH_DATE: timestamp (nullable = true)
 |-- INJURIES_TOTAL: integer (nullable = true)
 |-- INJURIES_FATAL: integer (nullable = true)
 |-- MOST_SEVERE_INJURY: string (nullable = true)
 |-- WEATHER_CONDITION: string (nullable = true)
 |-- LIGHTING_CONDITION: string (nullable = true)
 |-- ROADWAY_SURFACE_COND: string (nullable = true)
 |-- PRIM_CONTRIBUTORY_CAUSE: string (nullable = true)
 |-- SEC_CONTRIBUTORY_CAUSE: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)

+----------------------------------------+-------------------+--------------+--------------+------------------------+-----------------+------------------+--------------------+---------------------------+---------------------------+----+-----+
|                         CRASH_RECOR

## Step 4 — Write Silver Parquet

We write the cleaned dataset as **Parquet**, partitioned by `year` then `month`.

**Why Parquet?**
- Columnar format — reads only requested columns, skips the rest.
- Built-in compression — typically 5–10× smaller than CSV.
- Natively supported by all downstream Spark reads.

**Why partition by `year`/`month`?**
- Downstream batch queries filtered to a specific month read only that partition.
- The streaming replay script (`make_replay_files.py`) can chunk files by month without re-scanning the full dataset.

**`mode("overwrite")`** makes this step idempotent — re-running the notebook on the same snapshot replaces the previous Silver output cleanly.

> **Note:** After the write, Parquet stores partition keys (`year`, `month`) as directory names (`year=2022/month=1/`). Spark automatically re-attaches them as columns when reading back, so they are never lost.

In [5]:
print(f"Writing Silver Parquet to: {os.path.abspath(SILVER_PATH)}")
print("Partitioning by: year, month")
print("Mode: overwrite (idempotent re-runs)\n")

(
    silver_df
    .write
    .mode("overwrite")
    .partitionBy("year", "month")
    .parquet(SILVER_PATH)
)

print("✓ Silver write complete.")

Writing Silver Parquet to: c:\Users\miguel.morales\GIT\AI Certificate\Big Data\big-data_group-10\data\silver\crashes_parquet
Partitioning by: year, month
Mode: overwrite (idempotent re-runs)

✓ Silver write complete.


## Step 5 — Read-Back Validation

We immediately read the Silver Parquet back to confirm:

1. The schema is correct (`CRASH_DATE` is `timestamp`, injury columns are `int`, rest are `string`).
2. Row counts per year match expectations and cover only 2022–2025.
3. No year outside the scope appears in the output.

This is a **sanity check only** — not analysis. Pass/fail is determined by the assertions at the end of this cell.

In [6]:
# Read Silver back from disk
silver_check = spark.read.parquet(SILVER_PATH)

# ── Schema ────────────────────────────────────────────────────────────────────
print("=== Silver Schema ===")
silver_check.printSchema()

# ── Row count per year ────────────────────────────────────────────────────────
print("=== Row count by year ===")
silver_check.groupBy("year").count().orderBy("year").show()

# ── Top 5 rows ────────────────────────────────────────────────────────────────
print("=== Sample rows ===")
silver_check.show(5, truncate=40)

# ── Assertions ────────────────────────────────────────────────────────────────
silver_total = silver_check.count()
years_in_silver = {
    row["year"]
    for row in silver_check.select("year").distinct().collect()
}

assert silver_total == post_filter_count, (
    f"Row count mismatch: wrote {post_filter_count:,}, read back {silver_total:,}"
)
assert years_in_silver.issubset({2022, 2023, 2024, 2025}), (
    f"Unexpected years in Silver: {years_in_silver - {2022, 2023, 2024, 2025}}"
)

print(f"\n✓ All assertions passed.")
print(f"  Total Silver rows : {silver_total:,}")
print(f"  Years present     : {sorted(years_in_silver)}")

=== Silver Schema ===
root
 |-- CRASH_RECORD_ID: string (nullable = true)
 |-- CRASH_DATE: timestamp (nullable = true)
 |-- INJURIES_TOTAL: integer (nullable = true)
 |-- INJURIES_FATAL: integer (nullable = true)
 |-- MOST_SEVERE_INJURY: string (nullable = true)
 |-- WEATHER_CONDITION: string (nullable = true)
 |-- LIGHTING_CONDITION: string (nullable = true)
 |-- ROADWAY_SURFACE_COND: string (nullable = true)
 |-- PRIM_CONTRIBUTORY_CAUSE: string (nullable = true)
 |-- SEC_CONTRIBUTORY_CAUSE: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)

=== Row count by year ===
+----+------+
|year| count|
+----+------+
|2022|108412|
|2023|110755|
|2024|112055|
|2025|109108|
+----+------+

=== Sample rows ===
+----------------------------------------+-------------------+--------------+--------------+------------------------+-----------------+----------------------+--------------------+----------------------------+----------------------------------

In [7]:
# Stop the Spark session cleanly.
# Always stop when done to release resources in a local environment.
spark.stop()
print("Spark session stopped. Bronze → Silver ETL complete.")

Spark session stopped. Bronze → Silver ETL complete.
